# Scaling Laws for Neural Language Models — Implementation / 구현

**Paper**: Kaplan, J., McCandlish, S., Henighan, T., Brown, T. B., Chess, B., Child, R., Gray, S., Radford, A., Wu, J., & Amodei, D. (2020). Scaling Laws for Neural Language Models. *arXiv:2001.08361*.

## 목적 / Goal

**한국어**
이 노트북은 Kaplan et al. (2020)의 핵심 발견인 **모델 크기 $N$과 손실 $L$ 사이의 멱법칙(power law)** 관계를
토이 합성 작업에서 경험적으로 재현합니다. 다양한 크기의 작은 Transformer 같은 모델들을 합성 시퀀스 모델링
작업에 학습시키고, 손실을 매개변수 수의 함수로 측정한 뒤 멱법칙 $L(N) = (N_c/N)^{\alpha_N}$을 적합합니다.
측정한 $\alpha_N$을 논문의 $\alpha_N \approx 0.076$과 비교합니다 (실제 값은 작업의 단순함, 모델 규모, 데이터에
따라 달라짐).

**English**
This notebook empirically reproduces the **power-law relationship between model size $N$ and loss $L$** —
the central finding of Kaplan et al. (2020) — on a toy synthetic task. We train small Transformer-like
models of varying size on a synthetic sequence-modeling task, measure loss as a function of parameter count,
and fit a power law $L(N) = (N_c/N)^{\alpha_N}$. We compare our measured exponent to the paper's
$\alpha_N \approx 0.076$ (note that toy-task exponents typically differ due to task simplicity and limited
scale range).

In [ ]:
# Imports and reproducibility setup
# 임포트 및 재현성 설정
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Part 1: Synthetic Sequence Modeling Task / 합성 시퀀스 모델링 작업

**한국어**
토이 자기회귀 언어 모델 작업을 만듭니다. 어휘 크기 $V=64$, 시퀀스 길이 $T=32$.
각 시퀀스는 랜덤 마르코프 체인으로 생성된 토큰들로, 다음 토큰은 이전 두 토큰에 의존합니다.
이 작업은 충분히 학습 가능하면서, 큰 모델일수록 손실이 줄어드는 멱법칙 영역을 갖습니다.

**English**
We construct a toy autoregressive language modeling task: vocabulary $V=64$, sequence length $T=32$.
Each sequence is generated from a random Markov chain where the next token depends on the previous two
tokens. The task is sufficiently learnable while exhibiting a power-law regime where bigger models give
lower loss.

In [ ]:
# Generate synthetic sequences from a random 2nd-order Markov chain
# 랜덤 2차 마르코프 체인에서 합성 시퀀스 생성
def make_markov_dataset(num_sequences: int, seq_len: int, vocab_size: int,
                       order: int = 2, seed: int = 42) -> torch.Tensor:
    """Generate sequences from a random higher-order Markov chain.

    Args:
        num_sequences: Number of sequences in the dataset.
        seq_len: Length of each sequence (in tokens).
        vocab_size: Number of distinct tokens.
        order: Order of the Markov chain (number of preceding tokens).
        seed: Random seed for reproducibility.

    Returns:
        Tensor of shape (num_sequences, seq_len) with integer token ids.
    """
    rng = np.random.default_rng(seed)
    # Build a sparse transition table: each (order)-tuple -> distribution over next token.
    # We use a low-entropy distribution (concentrated on a few tokens) so the task is learnable.
    n_states = vocab_size ** order
    logits = rng.standard_normal((n_states, vocab_size)) * 3.0
    probs = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = probs / probs.sum(axis=1, keepdims=True)

    sequences = np.zeros((num_sequences, seq_len), dtype=np.int64)
    sequences[:, :order] = rng.integers(0, vocab_size, size=(num_sequences, order))
    for t in range(order, seq_len):
        # Encode the previous (order) tokens as a single state index
        state_idx = np.zeros(num_sequences, dtype=np.int64)
        for k in range(order):
            state_idx = state_idx * vocab_size + sequences[:, t - order + k]
        # Sample next token from the transition distribution
        for i in range(num_sequences):
            sequences[i, t] = rng.choice(vocab_size, p=probs[state_idx[i]])
    return torch.from_numpy(sequences)

VOCAB_SIZE = 64
SEQ_LEN = 32
N_TRAIN = 4000
N_VAL = 1000

train_data = make_markov_dataset(N_TRAIN, SEQ_LEN, VOCAB_SIZE, order=2, seed=42)
val_data = make_markov_dataset(N_VAL, SEQ_LEN, VOCAB_SIZE, order=2, seed=43)
print(f'Train shape: {train_data.shape}, Val shape: {val_data.shape}')

# A trivial unigram baseline gives loss ~log(V) = 4.16 nats; the Markov structure should let us
# do better. The irreducible loss is bounded by the entropy of the transition distribution.
print(f'Uniform-baseline loss: {math.log(VOCAB_SIZE):.4f} nats')

## Part 2: Tiny Transformer Model / 작은 Transformer 모델

**한국어**
단순한 decoder-only Transformer를 정의합니다. 너비($d_{\text{model}}$)와 깊이($n_{\text{layer}}$)를 다양하게 변경하여
비임베딩 매개변수 수를 측정합니다 ($N \approx 12 \, n_{\text{layer}} \, d_{\text{model}}^2$, 논문 §2.1과 일치).

**English**
We define a simple decoder-only Transformer. We vary the width ($d_{\text{model}}$) and depth ($n_{\text{layer}}$)
and count non-embedding parameters ($N \approx 12 \, n_{\text{layer}} \, d_{\text{model}}^2$, matching the
paper's §2.1).

In [ ]:
class TinyTransformer(nn.Module):
    """A minimal decoder-only Transformer for autoregressive language modeling.

    Architecture follows Kaplan et al. (2020) §2.1 conventions:
    d_attn = d_ff/4 = d_model, so non-embedding params ~ 12 * n_layer * d_model^2.
    """

    def __init__(self, vocab_size: int, d_model: int, n_layer: int,
                 n_head: int, seq_len: int, dropout: float = 0.0):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_head,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=n_layer)
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.seq_len = seq_len
        # Causal mask (upper triangular True = masked positions)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        self.register_buffer('causal_mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Compute next-token logits.

        Args:
            x: Long tensor of shape (B, T) with token ids.

        Returns:
            Logits of shape (B, T, V).
        """
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        h = self.token_emb(x) + self.pos_emb(pos)
        h = self.transformer(h, mask=self.causal_mask[:T, :T])
        h = self.ln_f(h)
        return self.head(h)

    def non_embedding_params(self) -> int:
        """Count parameters excluding token/position embeddings and the unembedding head.

        Returns:
            Number of non-embedding parameters (Kaplan et al. §2.1 convention).
        """
        total = sum(p.numel() for p in self.parameters())
        embed = (self.token_emb.weight.numel() + self.pos_emb.weight.numel()
                 + self.head.weight.numel())
        return total - embed

# Quick sanity check on one model
model = TinyTransformer(VOCAB_SIZE, d_model=32, n_layer=2, n_head=4, seq_len=SEQ_LEN)
print(f'd_model=32, n_layer=2 -> non-embedding params: {model.non_embedding_params():,}')
approx = 12 * 2 * 32 ** 2
print(f'Theoretical 12 * n_layer * d_model^2 = {approx:,}')

## Part 3: Train Models of Varying Size / 다양한 크기의 모델 학습

**한국어**
여러 (`d_model`, `n_layer`) 조합으로 모델들을 학습시키고 검증 손실을 기록합니다. 각 모델은 동일한 데이터/스텝/optimizer로
학습됩니다. 매개변수 수 $N$을 두 자리 이상의 범위에서 변동시킵니다.

**English**
We train models with several (`d_model`, `n_layer`) combinations and record validation loss.
Each model is trained with the same data/steps/optimizer. We span $N$ over more than two orders of magnitude.

In [ ]:
def train_one(model: nn.Module, train_data: torch.Tensor, val_data: torch.Tensor,
              n_epochs: int = 8, batch_size: int = 64, lr: float = 3e-3,
              device: torch.device = device) -> float:
    """Train a model and return its final validation cross-entropy loss.

    Args:
        model: Model to train (modified in place).
        train_data: Tensor of shape (N_train, T) with integer token ids.
        val_data: Tensor of shape (N_val, T) with integer token ids.
        n_epochs: Number of full passes through the training set.
        batch_size: Mini-batch size.
        lr: Learning rate for AdamW.
        device: torch device for computation.

    Returns:
        Mean validation loss in nats per token over the validation set.
    """
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    n_steps_total = n_epochs * (len(train_data) // batch_size)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, n_steps_total))

    train_ds = TensorDataset(train_data)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

    model.train()
    for epoch in range(n_epochs):
        for (batch,) in train_loader:
            batch = batch.to(device)
            inputs = batch[:, :-1]
            targets = batch[:, 1:]
            logits = model(inputs)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   targets.reshape(-1))
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()

    # Validation
    model.eval()
    with torch.no_grad():
        v = val_data.to(device)
        inputs = v[:, :-1]
        targets = v[:, 1:]
        logits = model(inputs)
        val_loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   targets.reshape(-1)).item()
    return val_loss

# Sweep over a grid of (d_model, n_layer) configurations
configs = [
    # (d_model, n_layer, n_head)
    (16, 1, 2),
    (24, 1, 4),
    (32, 2, 4),
    (48, 2, 4),
    (64, 2, 4),
    (96, 2, 4),
    (64, 4, 4),
    (96, 4, 4),
    (128, 4, 8),
]

results = []  # list of (N_non_embed, val_loss, config)
for d_model, n_layer, n_head in configs:
    model = TinyTransformer(VOCAB_SIZE, d_model=d_model, n_layer=n_layer,
                            n_head=n_head, seq_len=SEQ_LEN)
    n_params = model.non_embedding_params()
    val_loss = train_one(model, train_data, val_data, n_epochs=8,
                         batch_size=64, lr=3e-3)
    print(f'd_model={d_model:3d}, n_layer={n_layer}, N={n_params:>8,}, val_loss={val_loss:.4f}')
    results.append((n_params, val_loss, (d_model, n_layer, n_head)))

results.sort(key=lambda x: x[0])
Ns = np.array([r[0] for r in results], dtype=np.float64)
Ls = np.array([r[1] for r in results], dtype=np.float64)

## Part 4: Fit the Power Law $L(N) = (N_c/N)^{\alpha_N}$ / 멱법칙 적합

**한국어**
본 논문 식 (1.1)을 사용해 손실을 매개변수 수의 함수로 적합합니다. 자연어 작업의 비-zero entropy를 고려해
$L(N) = L_\infty + (N_c/N)^{\alpha_N}$ 형태의 offset 적합도 시도합니다 (작은 작업이므로 실험적 보정).
또한 단순한 log-log 선형 회귀로 지수만 추정합니다.

**English**
We fit Eq. (1.1) of the paper. Because real-world tasks have non-zero entropy, we also try an offset fit
$L(N) = L_\infty + (N_c/N)^{\alpha_N}$. Additionally we estimate the exponent via simple log-log linear
regression.

In [ ]:
from scipy.optimize import curve_fit

def power_law(N: np.ndarray, Nc: float, alpha: float) -> np.ndarray:
    """Pure power law L(N) = (Nc/N)^alpha (Eq. 1.1 of Kaplan et al. 2020).

    Args:
        N: Array of parameter counts.
        Nc: Characteristic parameter scale.
        alpha: Power-law exponent.

    Returns:
        Predicted loss for each N.
    """
    return (Nc / N) ** alpha

def power_law_offset(N: np.ndarray, L_inf: float, Nc: float, alpha: float) -> np.ndarray:
    """Power law with an irreducible-loss offset, L(N) = L_inf + (Nc/N)^alpha.

    Args:
        N: Array of parameter counts.
        L_inf: Asymptotic loss (irreducible task entropy).
        Nc: Characteristic parameter scale.
        alpha: Power-law exponent.

    Returns:
        Predicted loss for each N.
    """
    return L_inf + (Nc / N) ** alpha

# (1) Pure power law fit
popt_pure, _ = curve_fit(power_law, Ns, Ls, p0=[1e6, 0.1], maxfev=10000)
Nc_pure, alpha_pure = popt_pure

# (2) log-log linear regression: log L = -alpha * log N + const
log_slope, log_intercept = np.polyfit(np.log(Ns), np.log(Ls), 1)
alpha_log = -log_slope

# (3) Offset power law (more physical given non-zero task entropy)
try:
    popt_off, _ = curve_fit(power_law_offset, Ns, Ls,
                            p0=[Ls.min() * 0.5, 1e6, 0.1], maxfev=20000)
    L_inf_off, Nc_off, alpha_off = popt_off
except Exception as e:
    print(f'Offset fit failed: {e}')
    L_inf_off, Nc_off, alpha_off = float('nan'), float('nan'), float('nan')

print('Pure power law L(N) = (Nc/N)^alpha:')
print(f'  alpha_N (pure)        = {alpha_pure:.4f}')
print(f'  Nc                    = {Nc_pure:.3e}')
print('log-log linear regression:')
print(f'  alpha_N (log-log)     = {alpha_log:.4f}')
print('Offset power law L(N) = L_inf + (Nc/N)^alpha:')
print(f'  alpha_N (offset)      = {alpha_off:.4f}')
print(f'  L_inf (irreducible)   = {L_inf_off:.4f}')
print(f'  Nc                    = {Nc_off:.3e}')
print()
print('Reference: Kaplan et al. 2020 report alpha_N ~ 0.076 for Transformer LMs on WebText2.')
print('Note: toy-task exponents typically differ — Markov tasks have low irreducible entropy and small N range.')

## Part 5: Visualize on Log-Log Axes / 로그-로그 축 시각화

**한국어**
$\log L$ vs $\log N$ plot에서 멱법칙은 직선으로 나타납니다. 적합 결과를 함께 그립니다.

**English**
On a log-log plot, a power law appears as a straight line. We overlay the fitted curve.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(Ns, Ls, s=80, color='C0', zorder=5, label='Trained models / 학습된 모델')

N_grid = np.logspace(np.log10(Ns.min() * 0.5), np.log10(Ns.max() * 2), 100)
ax.plot(N_grid, power_law(N_grid, *popt_pure),
        '--', color='C1', lw=2,
        label=f'Pure: $L=(N_c/N)^{{\\alpha}}$, $\\alpha$={alpha_pure:.3f}')
if not np.isnan(alpha_off):
    ax.plot(N_grid, power_law_offset(N_grid, *popt_off),
            '-.', color='C2', lw=2,
            label=f'Offset: $L_\\infty$+$(N_c/N)^{{\\alpha}}$, $\\alpha$={alpha_off:.3f}')

# Paper's reference exponent (renormalized to our data range)
alpha_paper = 0.076
Nc_paper = Ns[len(Ns) // 2] * (Ls[len(Ls) // 2]) ** (1.0 / alpha_paper)
ax.plot(N_grid, (Nc_paper / N_grid) ** alpha_paper,
        ':', color='C3', lw=2,
        label=f"Kaplan et al. slope $\\alpha$={alpha_paper}")

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Non-embedding parameters $N$ / 비임베딩 매개변수')
ax.set_ylabel('Validation loss $L$ (nats/token) / 검증 손실')
ax.set_title('Empirical scaling law on a toy Markov task / 토이 마르코프 작업의 경험적 스케일링 법칙')
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Part 6: Comparison to Paper / 논문과의 비교

**한국어**
측정한 $\alpha_N$을 논문 값과 비교하고, 차이의 원인을 분석합니다.

**English**
We compare our measured $\alpha_N$ to the paper's value and discuss the discrepancy.

In [ ]:
comparison = [
    ('Pure power law / 순수 멱법칙', alpha_pure),
    ('log-log linear / 로그-로그 선형', alpha_log),
    ('Offset power law / 오프셋 멱법칙', alpha_off),
    ('Kaplan et al. (2020) WebText2', 0.076),
    ('Hoffmann et al. (2022) Chinchilla', 0.34),  # roughly, with corrected scaling
]

print('=' * 60)
print('Estimated alpha_N values / 추정한 alpha_N 값')
print('=' * 60)
for name, val in comparison:
    print(f'  {name:45s} {val:.4f}')
print('=' * 60)

print()
print('Why our exponent differs / 지수가 다른 이유:')
print('1. Toy Markov task has low irreducible entropy; the offset fit is more meaningful.')
print('   토이 마르코프 작업은 entropy가 낮아 offset 적합이 더 의미 있음.')
print('2. We span only ~2 orders of magnitude in N; the paper spans 6+.')
print('   본 노트북은 N을 약 2자리만 변동; 논문은 6자리 이상.')
print('3. Limited training (8 epochs) means smaller models are not at their L(N, infty).')
print('   짧은 학습 (8 epochs)은 작은 모델이 L(N, infty)에 도달 못함을 의미.')
print('4. The paper studies natural language; the structure of natural-language entropy')
print('   gives a cleaner power-law regime.')
print('   자연어 entropy 구조가 더 깨끗한 멱법칙 영역을 만듦.')

## Summary / 요약

**한국어**
- 토이 마르코프 작업에서 모델 크기와 검증 손실 사이의 멱법칙을 경험적으로 관찰.
- 비임베딩 매개변수 수 $N$을 약 2자리 변동시켜 $L(N) = (N_c/N)^{\alpha_N}$ 적합.
- 측정된 $\alpha_N$이 논문의 0.076과 정확히 일치하지는 않으나, **손실이 $N$에 단조 감소하며 멱법칙 추세를 따른다**는
  핵심 메시지를 재현.
- offset 적합 ($L_\infty + (N_c/N)^\alpha$)이 작은 작업에서는 더 적절 — 이는 Kaplan et al. 자체도 §6.3에서
  자연어의 entropy 한계 ($L^* \sim 1.7$)를 추측한 것과 동일한 형태.

**English**
- We empirically observe a power-law trend between model size and validation loss on a toy Markov task.
- We span $N$ over ~2 orders of magnitude and fit $L(N) = (N_c/N)^{\alpha_N}$.
- Our measured $\alpha_N$ does not exactly match the paper's 0.076, but **the qualitative finding — loss
  decreases monotonically as a power-law in $N$ — is reproduced**.
- The offset fit ($L_\infty + (N_c/N)^\alpha$) is more appropriate at small scale, mirroring Kaplan et al.'s
  own §6.3 conjecture about a natural-language entropy bound ($L^* \sim 1.7$).

## Mapping to Paper / 논문과의 매핑

| Concept / 개념 | This Notebook / 이 노트북 | Paper / 논문 |
|---|---|---|
| Architecture / 아키텍처 | TinyTransformer ($d \le 128$, $L \le 4$) | Transformer ($d \le 6400$, $L \le 207$) |
| Task / 작업 | 2nd-order Markov chain, $V=64$, $T=32$ | Autoregressive LM on WebText2 |
| Data scale / 데이터 규모 | 4000 sequences | $2.29 \times 10^{10}$ tokens |
| $N$ range | $\sim 10^3 - 10^5$ | $10^3 - 10^9$ |
| Power law / 멱법칙 | $L = (N_c/N)^{\alpha}$ (Eq. 1.1) | $L = (N_c/N)^{\alpha_N}$ (Eq. 1.1) |
| Reported $\alpha$ / 보고된 지수 | varies (toy) | $\alpha_N \approx 0.076$ |
| Joint $L(N,D)$ | (not implemented here) | $[(N_c/N)^{\alpha_N/\alpha_D} + D_c/D]^{\alpha_D}$ |

## Caveat: Chinchilla / 주의: Chinchilla

**한국어**
Hoffmann et al. (2022) — "Chinchilla" — 는 Kaplan et al.의 학습률 스케줄이 작은 모델에 불리하게 설정되어
$\alpha_N$이 과대 추정됐다고 주장합니다. 정정 후 비율은 "모델과 데이터를 1:1로 함께 키워야" 한다는 결과로 이어졌으며,
이는 GPT-3가 "undertrained"였다는 후속 분석의 근거가 됐습니다.

**English**
Hoffmann et al. (2022) — "Chinchilla" — argued that Kaplan et al.'s learning-rate schedule
disadvantaged smaller models, biasing $\alpha_N$ upward. The corrected analysis gave a 1:1 model:data
scaling, leading to the conclusion that GPT-3 was "undertrained".